In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Cell 1 — Install dependencies

In [1]:
!pip install -q ultralytics roboflow
!pip uninstall -y ray


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Found existing installation: ray 2.54.0
Uninstalling ray-2.54.0:
  Successfully uninstalled ray-2.54.0


Cell 2 — Imports & GPU check

In [5]:
from ultralytics import YOLO
from roboflow import Roboflow
import torch
import os
import yaml

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("⚠ No GPU detected. Enable GPU in Kaggle: Settings → Accelerator → GPU P100")

GPU Available: True
GPU Name: Tesla T4
VRAM (GB): 15.64


Cell 3 — Download dataset

In [ ]:
rf = Roboflow(api_key="bXaEda73XF552aWMibWp")  # 🔴 Replace with your Roboflow API key

project = rf.workspace("visheshs-workspace").project("islfinaldataset")

# Try yolov11 format; falls back to yolov8 (same format, fully compatible)
try:
    dataset = project.version(4).download("yolov11")
except:
    dataset = project.version(4).download("yolov8")

data_yaml_path = f"{dataset.location}/data.yaml"
print("Dataset path:", data_yaml_path)

Cell 4 — Dataset analysis

In [8]:
for split in ["train", "valid", "test"]:
    path = f"{dataset.location}/{split}/images"
    if os.path.exists(path):
        print(f"{split}: {len(os.listdir(path))} images")
    else:
        print(f"{split}: not found")

# Check class count and names from data.yaml
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print(f"\nClasses ({data_cfg['nc']}):", data_cfg['names'])

# Check class balance in train labels
label_dir = f"{dataset.location}/train/labels"
class_counts = {}
for lf in os.listdir(label_dir):
    with open(f"{label_dir}/{lf}") as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts[cls] = class_counts.get(cls, 0) + 1

print("\nClass distribution (train):")
for cls_id, count in sorted(class_counts.items()):
    name = data_cfg['names'][cls_id]
    print(f"  {name}: {count}")

train: 17663 images
valid: 1017 images
test: 288 images

Classes (123): ['0', '1', '2', '3', '4', '5', '6', '8', '9', 'A', 'B', 'Bad', 'Band Aid', 'Born', 'Brother', 'Bye', 'C', 'Cough', 'D', 'E', 'Eat', 'Eight', 'F', 'Father', 'Five', 'Food', 'Four', 'Friend', 'G', 'Good', 'H', 'Hello', 'Help', 'Home', 'House', 'I', 'I-Love-You', 'Indian', 'J', 'K', 'L', 'Like', 'Loud', 'Love', 'M', 'Mummy', 'N', 'Namaste', 'Name', 'Nine', 'No', 'O', 'One', 'P', 'Peace', 'Place', 'Please', 'Q', 'Quiet', 'R', 'Request', 'S', 'Seven', 'Six', 'Sleeping', 'Sorry', 'Stop', 'Strong', 'T', 'Ten', 'Thank-you', 'Three', 'Time', 'Today', 'Two', 'U', 'V', 'W', 'Water', 'What', 'When', 'X', 'Y', 'Yes', 'Your', 'Z', 'Zero', 'blue', 'college', 'drink', 'drive', 'food', 'friday', 'green', 'hii', 'home', 'how are you', 'language', 'me', 'monday', 'namaste', 'okay', 'orange', 'pink', 'purple', 'red', 'salute', 'school', 'sick', 'sign', 'sleep', 'smile', 'stand', 'stop', 'strong', 'studyy', 'sunday', 'thursday', 'tuesd

Cell 5 — Load model

In [7]:
model = YOLO("yolo11m.pt")

Cell 6 — Train model

In [11]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/datasets/visheshkamble18/finalisl/last (23).pt")

results = model.train(
    data=data_yaml_path,
    epochs=50,          # recommended
    imgsz=640,
    batch=16,
    device=0,
    workers=4,

    optimizer="auto",
    patience=30,
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,

    project="isl_training",
    name="yolov11m_isl_final",

    cache=True,
    amp=True,
    verbose=True,

    resume=True   #
)


Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ISLFINALDATASET-4/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/visheshkamble18/finalisl/last (23).pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11m_isl_final, nbs=64, nms=False, opset=None, opt

KeyboardInterrupt: 

Cell 7 — Evaluate model

In [12]:
metrics = model.val()

print("\n=== Validation Results ===")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,124,865 parameters, 0 gradients, 68.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 834.7±289.1 MB/s, size: 23.4 KB)
val: Scanning /kaggle/working/ISLFINALDATASET-4/valid/labels.cache... 1017 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1017/1017 426.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 1.8it/s 36.1s0.6ss
                   all       1017       1024      0.966      0.973      0.989      0.818
                     0          4          4      0.972          1      0.995      0.798
                     1          9          9      0.977          1      0.995      0.863
                     2          2          2      0.916          1      0.995      0.854
                     4          1          1      0.879          1      0.995      0.895
                  

Cell 8 — Test predictions

In [ ]:
# Use test set if it exists, otherwise use valid set
test_path = f"{dataset.location}/test/images"
predict_source = test_path if os.path.exists(test_path) else f"{dataset.location}/valid/images"

results = model.predict(
    source=predict_source,
    conf=0.5,
    save=True,
    project="isl_predictions",
    name="final_test"
)

print(f"Predictions saved to isl_predictions/final_test/")

Cell 9 — Download best model

In [13]:
from IPython.display import FileLink
import shutil

best_model_path = "isl_training/yolov11m_isl_final/weights/best.pt"

if os.path.exists(best_model_path):
    print("✅ Model ready for download:")
    display(FileLink(best_model_path))

    # Also copy to /kaggle/working/ for easy access
    shutil.copy(best_model_path, "/kaggle/working/isl_best.pt")
    print("Also copied to /kaggle/working/isl_best.pt")
else:
    print("⚠ Model not found. Check if training completed successfully.")

⚠ Model not found. Check if training completed successfully.


Download Model metrices 

In [14]:
import os
import shutil
from IPython.display import FileLink

# Source folder (your training output)
source_dir = "/kaggle/working/runs/detect/isl_training/yolov11m_isl_final"

# Optional: copy best model separately for safety
best_model = os.path.join(source_dir, "weights/best.pt")
if os.path.exists(best_model):
    shutil.copy(best_model, "/kaggle/working/isl_best.pt")

# Create ZIP file
zip_path = "/kaggle/working/isl_complete_project"
shutil.make_archive(zip_path, 'zip', source_dir)

print("ZIP created successfully!")

# Download link
FileLink(zip_path + ".zip")


ZIP created successfully!


/kaggle/working/isl_complete_project.zip

resume 

In [6]:
from ultralytics import YOLO

# Load last checkpoint
model = YOLO("/kaggle/input/datasets/visheshkamble18/lastptv1")

# Resume training
results = model.train(
    data=data_yaml_path,
    epochs=100,          # total epochs 
    imgsz=640,
    batch=16,
    device=0,
    workers=4,

    optimizer="auto",
    patience=30,
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,

    # Augmentation — tuned for hand signs
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,

    project="isl_training",
    name="yolov11m_isl_final",

    cache=True,
    amp=True,
    verbose=True,

    resume=True   
)


WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.


TypeError: model='/kaggle/input/datasets/visheshkamble18/lastptv1' should be a *.pt PyTorch model to run this method, but is a different format. PyTorch models can train, val, predict and export, i.e. 'model.train(data=...)', but exported formats like ONNX, TensorRT etc. only support 'predict' and 'val' modes, i.e. 'yolo predict model=yolo26n.onnx'.
To run CUDA or MPS inference please pass the device argument directly in your inference command, i.e. 'model.predict(source=..., device=0)'